# Solutions: Unit Testing Exercises

This notebook provides step-by-step solutions for writing and running unit tests in your RAP pipeline using pytest. Each solution matches the corresponding exercise notebook and is designed for beginners.

## Exercise 1 solution: Write a simple unit test for a new function

Here are example unit tests for the `flag_missing` and `impute_by_group` functions in `src/python_rap_demo/cleaning.py`:

In [ ]:
# Walkthrough: Unit test for flag_missing
import pandas as pd

from python_rap_demo.cleaning import flag_missing


def test_flag_missing():
    """
    Test flag_missing
    """
    df = pd.DataFrame({"height_cm": [170, None], "weight_kg": [70, None]})
    flagged = flag_missing(df, ["height_cm", "weight_kg"])
    # Check that the _imputed columns are correct
    assert flagged["height_cm_imputed"].tolist() == [False, True]
    assert flagged["weight_kg_imputed"].tolist() == [False, True]

In [ ]:
# Example: Unit test for impute_by_group
import pandas as pd

from python_rap_demo.cleaning import impute_by_group


def test_impute_by_group():
    """
    Test impute_by_group.
    """
    df = pd.DataFrame({"height_cm": [170, None, 160], "sex": ["M", "F", "F"]})
    imputed = impute_by_group(df, "height_cm", "sex")
    # Check that missing value is imputed with group mean
    expected = [170, 160, 160]
    assert imputed.tolist() == expected

## Exercise 2 solution: Run your unit tests

Run the following command in your terminal:
```cmd
pytest tests
```
**Expected output:**
- All tests should pass. If a test fails, check the error message and fix your code or tests.

## Exercise 3 solution: Stretch - Check test coverage

Run the following commands:
```cmd
pip install coverage
coverage run -m pytest tests
coverage report
```
**Expected output:**
- You will see a report showing the percentage of code covered by tests. Aim for high coverage, but focus on testing important logic.

## Exercise 4 solution: Stretch - Try parameterisation in pytest

Here are examples using `@pytest.mark.parametrize` for `flag_missing` and `impute_by_group`. Parameterisation lets you run the same test with different inputs, making your tests more robust and easier to maintain.

In [ ]:
import pandas as pd
import pytest

from python_rap_demo.cleaning import flag_missing, impute_by_group

# Parameterised test for flag_missing


@pytest.mark.parametrize(
    "df,columns,expected_height_flags,expected_weight_flags",
    [
        # Test case 1: One missing value in each column
        (
            pd.DataFrame({"height_cm": [170, None], "weight_kg": [70, None]}),
            ["height_cm", "weight_kg"],
            [False, True],
            [False, True],
        ),
        # Test case 2: All missing in height, one missing in weight
        (
            pd.DataFrame({"height_cm": [None, None], "weight_kg": [None, 80]}),
            ["height_cm", "weight_kg"],
            [True, True],
            [True, False],
        ),
    ],
)
def test_flag_missing_param(df, columns, expected_height_flags, expected_weight_flags):
    """
    Test flag_missing with multiple input cases using parameterisation.
    """
    flagged = flag_missing(df, columns)
    # Check that the _imputed columns match expected flags
    assert flagged["height_cm_imputed"].tolist() == expected_height_flags
    assert flagged["weight_kg_imputed"].tolist() == expected_weight_flags


# Parameterised test for impute_by_group


@pytest.mark.parametrize(
    "df,col,group_col,expected",
    [
        # Test case 1: Impute missing height by sex group mean
        (
            pd.DataFrame({"height_cm": [170, None, 160], "sex": ["M", "F", "F"]}),
            "height_cm",
            "sex",
            [170, 160, 160],
        ),
        # Test case 2: All missing in one group, fallback to overall mean
        (
            pd.DataFrame({"height_cm": [None, None, 150], "sex": ["M", "M", "F"]}),
            "height_cm",
            "sex",
            [150, 150, 150],
        ),
    ],
)
def test_impute_by_group_param(df, col, group_col, expected):
    """
    Test impute_by_group with multiple input cases using parameterisation.
    """
    imputed = impute_by_group(df, col, group_col)
    # Check that imputed values match expected output
    assert imputed.tolist() == expected